# Point2MAE training — slope units with XY positional embeddings

This notebook trains the **local Point-M2AE** implementation in
`/home/khan365/Home_code/Point-M2AE-main/Point-M2AE-main` using LAS/LAZ clouds under
`Output/full_20261006_001322_611477/All_Slope_Units`.

Before splitting, it removes every **district + slope-unit ID** present in either
`LS_head_scarp_Slope_units` or `LS_Slope_units_without_head_scarp`. None of those
units can enter training, validation, or testing. Windows `:Zone.Identifier`
sidecars are ignored. One cloud represents one slope unit. Every original point
is retained in each accepted cloud: there is no input downsampling, cropping,
or duplicate-point padding. The three eligible clouds with fewer than eight
points are excluded and recorded in `too_small_clouds.csv`, as requested.

Every encoder and decoder patch-centre positional embedding takes **X/Y only**.
The reconstruction head and original masked **XYZ Chamfer-L2** objective remain:
Z is still reconstructed as part of the masked local geometry. Visible point
features, FPS/KNN grouping, and decoder interpolation retain the original XYZ
geometry. This is not Z-only regression or a claim that every model operation
is independent of Z. The targets are centre-relative patches, not absolute
georeferenced elevations. Full-cloud input does not turn this into dense
reconstruction of every original point: the model still forms an internal
FPS/KNN hierarchy and reconstructs masked local neighbourhoods of its centres.

**Environment installation is deferred.** No cell installs packages or launches
an external training process. The settings and split-preparation cells use only
the Python standard library and can run before installation. Later cells require
a configured CUDA environment. Run cells in order; the training cell starts
training when explicitly executed (or when using Run All).

## 1. Choose training parameters

Run the next cell and answer its prompts; Enter accepts the displayed default.
Set `INTERACTIVE = False` to edit/use the defaults without prompts. Defaults are
starting values, not tuned hyperparameters. Each execution creates a new run
folder to avoid overwriting an earlier run. The original architecture uses three
encoder scales and two decoder scales; group counts now come from `group_divisors`, not fixed patch counts.
With `[8, 2, 4]`, use `ceil(N/8)`, then `ceil(G1/2)`, then `ceil(G2/4)`.
For example, 2,048 points produce 256 → 128 → 32 groups. Small clouds retain
the decoder minima `[8, 4, 3]`. `group_sizes` specifies KNN neighbours inside
each group, separately from the number of groups. The channel sizes/depths
below retain the supplied configuration. Larger clouds use more groups and GPU
memory; increase the divisors if necessary. No input points are dropped.

The default split is 85% train / 10% validation / 5% test, randomly by slope unit.
It is not a spatial or district holdout; nearby terrain can occur in different
splits. Complete validation/test clouds and fixed masks are used each epoch.
Each forward pass contains **one complete cloud**, so different point counts
need no padding. `accumulation_steps` sets the number of clouds averaged per
optimizer update (default 8); the last partial window is averaged correctly.
BatchNorm still sees one cloud at a time, so accumulation is not identical to
an eight-cloud forward pass. A single CUDA
GPU and full float32 precision are used because the supplied native extensions
expect CUDA float tensors. `num_workers=0` avoids notebook worker/pickling issues.
A conservative patch-size check reserves masked reconstruction targets even
when visible top-scale neighbourhoods do not overlap. Some custom settings are
rejected to guarantee that the reconstruction loss has targets.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import math
import uuid
import sys

INTERACTIVE = True
WORKSPACE = Path('/home/khan365/Point-M2AE-main').resolve()
sys.path.insert(0, str(WORKSPACE))
import point2mae_grouping
from point2mae_grouping import hierarchy_for_cloud

MODEL_ROOT = WORKSPACE / 'Point-M2AE-main'
DATA_ROOT = WORKSPACE / 'Output/full_20261006_001322_611477'
ALL_CLOUDS = DATA_ROOT / 'All_Slope_Units'
EXCLUSION_DIRS = [DATA_ROOT / 'LS_head_scarp_Slope_units',
                  DATA_ROOT / 'LS_Slope_units_without_head_scarp']

DEFAULTS = {
    'epochs': 100, 'accumulation_steps': 8,
    'learning_rate': 0.0001, 'weight_decay': 0.05, 'mask_ratio': 0.8,
    'train_fraction': 0.85, 'val_fraction': 0.1, 'seed': 42,
    'gradient_clip': 1.0, 'patience': 20, 'cuda_device': 0,
    'group_divisors': [8, 2, 4], 'group_sizes': [8, 4, 4],
}
LABELS = {
    'epochs': 'Maximum epochs', 'accumulation_steps': 'Full clouds per optimizer update',
    'learning_rate': 'AdamW learning rate',
    'weight_decay': 'AdamW weight decay', 'mask_ratio': 'Top-scale masking ratio',
    'train_fraction': 'Training fraction', 'val_fraction': 'Validation fraction',
    'seed': 'Random seed', 'gradient_clip': 'Gradient norm limit (0 disables)',
    'patience': 'Early stopping patience (0 disables)', 'cuda_device': 'CUDA device index',
    'group_divisors': 'Group-count divisors [n, d2, d3]: N/n, then G1/d2, then G2/d3 (JSON list)',
    'group_sizes': 'KNN neighbours per group, not the number of groups (JSON list)',
}

def ask(name, default):
    if not INTERACTIVE:
        return default
    while True:
        raw = input(f'{LABELS[name]} [{json.dumps(default)}]: ').strip()
        try:
            return default if not raw else (json.loads(raw) if isinstance(default, list) else type(default)(raw))
        except (ValueError, TypeError):
            print('Enter a number, or a JSON list such as [8, 2, 4].')

CFG = {name: ask(name, value) for name, value in DEFAULTS.items()}
for key in ('epochs', 'accumulation_steps'):
    if CFG[key] < 1:
        raise ValueError(f'{key} must be positive; edit settings and rerun.')
for key in ('seed', 'patience', 'cuda_device'):
    if CFG[key] < 0:
        raise ValueError(f'{key} must be nonnegative.')
if CFG['seed'] >= 2**32:
    raise ValueError('seed must be smaller than 2**32.')
for key in ('learning_rate', 'weight_decay', 'gradient_clip', 'mask_ratio', 'train_fraction', 'val_fraction'):
    if not math.isfinite(CFG[key]):
        raise ValueError(f'{key} must be finite.')
if CFG['learning_rate'] <= 0 or min(CFG['weight_decay'], CFG['gradient_clip']) < 0:
    raise ValueError('Learning rate must be positive; weight decay and clipping nonnegative.')
if not (0 < CFG['train_fraction'] < 1 and 0 < CFG['val_fraction'] < 1
        and CFG['train_fraction'] + CFG['val_fraction'] < 1):
    raise ValueError('All three split fractions must be positive and sum to 1.')
# Validate grouping for the smallest accepted cloud; preflight checks every actual cloud.
hierarchy_for_cloud(8, CFG['group_divisors'], CFG['group_sizes'], CFG['mask_ratio'])
if not (MODEL_ROOT / 'models/Point_M2AE.py').is_file():
    raise FileNotFoundError(f'Local model not found: {MODEL_ROOT}')
CFG['input_mode'] = 'full_cloud'
CFG['batch_size'] = 1
CFG['minimum_cloud_points'] = 8
CFG['test_fraction'] = 1 - CFG['train_fraction'] - CFG['val_fraction']
RUN_DIR = WORKSPACE / 'training_runs' / (
    'pointm2ae_xy_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8])
print(json.dumps(CFG, indent=2))
print('Artifacts will be written to:', RUN_DIR)

{
  "epochs": 100,
  "accumulation_steps": 8,
  "learning_rate": 0.0001,
  "weight_decay": 0.05,
  "mask_ratio": 0.8,
  "train_fraction": 0.85,
  "val_fraction": 0.1,
  "seed": 42,
  "gradient_clip": 1.0,
  "patience": 20,
  "cuda_device": 0,
  "group_divisors": [
    8,
    2,
    4
  ],
  "group_sizes": [
    8,
    4,
    4
  ],
  "input_mode": "full_cloud",
  "batch_size": 1,
  "minimum_cloud_points": 8,
  "test_fraction": 0.05000000000000002
}
Artifacts will be written to: /home/khan365/Home_code/Point-M2AE-main/training_runs/pointm2ae_xy_20261030T190853Z_d323ddc4


In [ ]:
import sys; print(sys.executable) 

## 2. Exclude landslide units, then split

Identity is the cloud's relative district path plus `SU_<number>` (extension
independent), so a LAS/LAZ copy cannot bypass exclusion and equal IDs in distinct
districts remain distinct. Missing or empty exclusion directories stop the run;
this prevents silently training without the exclusions. Duplicate files for one
unit also stop the run. `excluded.csv` records the matched exclusion sources.

Manifests record paths and IDs, not point arrays. No source clouds are changed.
At notebook creation, the filename inventory was 28,464 total, 3,223 excluded,
and **25,241 eligible before the small-cloud audit**. Header inspection identifies
three of those with 1, 2 and 7 points. Per your instruction these are recorded in
`too_small_clouds.csv` and excluded before splitting, leaving **25,238 clouds**.
No points are removed from any retained cloud. The cells recompute all counts
from the files currently on disk. `cloud_inventory.csv` records full point counts
and the per-cloud hierarchy. Other incompatible hierarchy settings stop preflight
instead of silently excluding additional clouds.

In [2]:
from pathlib import Path
import random
import re

def cloud_index(directory):
    directory = Path(directory).resolve()
    if not directory.is_dir():
        raise FileNotFoundError(f'Required cloud directory missing: {directory}')
    result = {}
    for path in sorted(directory.rglob('*')):
        if not path.is_file() or path.suffix.lower() not in {'.las', '.laz'}:
            continue
        relative = path.relative_to(directory)
        match = re.fullmatch(r'SU_(\d+)', path.stem, flags=re.IGNORECASE)
        if match is None or relative.parent == Path('.'):
            raise ValueError(f'Expected district/SU_<id>.las or .laz: {path}')
        key = relative.parent.as_posix() + '/SU_' + str(int(match.group(1)))
        if key in result:
            raise ValueError(f'Duplicate slope unit {key}: {result[key]} and {path}')
        result[key] = str(path.resolve())
    if not result:
        raise ValueError(f'No LAS/LAZ clouds found in required directory: {directory}')
    return result

def discover_eligible(all_directory, exclusion_directories):
    if not exclusion_directories:
        raise ValueError('Exclusion directories are required.')
    all_files = cloud_index(all_directory)
    sources = {}
    counts = {}
    for directory in exclusion_directories:
        index = cloud_index(directory)
        counts[str(directory)] = len(index)
        for key in index:
            sources.setdefault(key, []).append(Path(directory).name)
    eligible, excluded = [], []
    for key, path in sorted(all_files.items()):
        row = {'unit_key': key, 'district': key.rsplit('/', 1)[0], 'path': path}
        if key in sources:
            excluded.append(dict(row, exclusion_sources=';'.join(sorted(sources[key]))))
        else:
            eligible.append(row)
    audit = {'all_clouds': len(all_files), 'exclusion_directory_counts': counts,
             'excluded_clouds': len(excluded), 'eligible_clouds': len(eligible),
             'excluded_ids_missing_from_all': sorted(set(sources) - set(all_files))}
    assert not ({r['unit_key'] for r in eligible} & set(sources))
    return eligible, excluded, audit

def make_splits(rows, train_fraction, val_fraction, seed):
    if not (0 < train_fraction < 1 and 0 < val_fraction < 1 and train_fraction + val_fraction < 1):
        raise ValueError('Train, validation and test fractions must all be positive.')
    rows = sorted(rows, key=lambda row: row['unit_key'])
    if len({r['unit_key'] for r in rows}) != len(rows):
        raise ValueError('Duplicate unit IDs in eligible rows.')
    n_train, n_val = int(len(rows) * train_fraction), int(len(rows) * val_fraction)
    if min(n_train, n_val, len(rows) - n_train - n_val) < 1:
        raise ValueError('Too few clouds for three nonempty splits with these fractions.')
    random.Random(seed).shuffle(rows)
    splits = {'train': rows[:n_train], 'validation': rows[n_train:n_train + n_val],
              'test': rows[n_train + n_val:]}
    keys = [set(r['unit_key'] for r in split) for split in splits.values()]
    assert not (keys[0] & keys[1] or keys[0] & keys[2] or keys[1] & keys[2])
    assert len(set.union(*keys)) == len(rows)
    return splits

In [3]:
import struct
from point2mae_grouping import hierarchy_for_cloud
from pathlib import Path

def las_point_count(path):
    """Read the LAS/LAZ header only; no point decompression or extra package needed."""
    with Path(path).open('rb') as handle:
        header = handle.read(375)
    if len(header) < 227 or header[:4] != b'LASF':
        raise ValueError(f'Invalid or truncated LAS/LAZ header: {path}')
    major, minor = header[24], header[25]
    if major != 1:
        raise ValueError(f'Unsupported LAS version {major}.{minor}: {path}')
    header_size = struct.unpack_from('<H', header, 94)[0]
    if header_size < 227:
        raise ValueError(f'Invalid LAS header size: {path}')
    legacy = struct.unpack_from('<I', header, 107)[0]
    if minor >= 4:
        if header_size < 375 or len(header) < 375:
            raise ValueError(f'Truncated LAS 1.4+ header: {path}')
        extended = struct.unpack_from('<Q', header, 247)[0]
        return extended if extended else legacy
    return legacy

def audit_full_clouds(rows, cfg):
    ready, too_small, inventory = [], [], []
    for row in rows:
        count = las_point_count(row['path'])
        counted = dict(row, point_count=count)
        if count < cfg['minimum_cloud_points']:
            too_small.append(dict(counted, reason='Fewer than 8 original points; no duplication allowed'))
            continue
        try:
            groups, sizes = hierarchy_for_cloud(count, cfg['group_divisors'], cfg['group_sizes'], cfg['mask_ratio'])
        except ValueError as exc:
            raise ValueError(f'Incompatible hierarchy for {row["path"]} ({count} points): {exc}') from exc
        ready.append(counted)
        inventory.append(dict(counted, num_groups=groups, group_sizes=sizes))
    return ready, too_small, inventory

def accumulation_window_size(index, total, steps):
    """Divisor for a cloud's loss, including a shorter final accumulation window."""
    if steps < 1 or total < 1 or not 0 <= index < total:
        raise ValueError('Invalid accumulation window arguments.')
    start = (index // steps) * steps
    return min(steps, total - start)

In [4]:
import csv
import hashlib
from collections import Counter

eligible, excluded, audit = discover_eligible(ALL_CLOUDS, EXCLUSION_DIRS)
eligible, too_small, inventory = audit_full_clouds(eligible, CFG)
splits = make_splits(eligible, CFG['train_fraction'], CFG['val_fraction'], CFG['seed'])
RUN_DIR.mkdir(parents=True, exist_ok=False)

def write_csv(path, rows, fields):
    with Path(path).open('w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)

for split_name, rows in splits.items():
    write_csv(RUN_DIR / f'{split_name}.csv', rows, ['unit_key', 'district', 'path', 'point_count'])
    print(f'{split_name:12s}: {len(rows):6d} units; districts: {dict(Counter(r["district"] for r in rows))}')
write_csv(RUN_DIR / 'excluded.csv', excluded, ['unit_key', 'district', 'path', 'exclusion_sources'])
write_csv(RUN_DIR / 'too_small_clouds.csv', too_small,
          ['unit_key', 'district', 'path', 'point_count', 'reason'])
write_csv(RUN_DIR / 'cloud_inventory.csv',
          [dict(row, num_groups=json.dumps(row['num_groups']), group_sizes=json.dumps(row['group_sizes']))
           for row in inventory],
          ['unit_key', 'district', 'path', 'point_count', 'num_groups', 'group_sizes'])
audit['too_small_clouds'] = len(too_small)
audit['retained_full_clouds'] = len(eligible)
audit['retained_original_points'] = sum(row['point_count'] for row in eligible)
print('Too-small clouds:', [(row['unit_key'], row['point_count']) for row in too_small])
MODEL_CONFIG = dict(
    NAME='Point_M2AE', mask_ratio=CFG['mask_ratio'],
    group_sizes=CFG['group_sizes'], group_divisors=CFG['group_divisors'],
    encoder_depths=[5, 5, 5], encoder_dims=[96, 192, 384],
    local_radius=[0.32, 0.64, 1.28], decoder_depths=[1, 1],
    decoder_dims=[384, 192], decoder_up_blocks=[1, 1],
    drop_path_rate=0.1, num_heads=6,
)
RUN_CONFIG = {'training': CFG, 'model': MODEL_CONFIG, 'positional_coordinates': ['x', 'y'],
              'reconstruction': 'upstream masked local XYZ ChamferDistanceL2',
              'normalization': 'subtract cloud XYZ centroid only; no rescaling',
              'input_sampling': 'none: every original point, batch_size=1',
              'hierarchy_policy': 'successive ceiling division by group_divisors; minima [8, 4, 3]; mask-safe neighbours',
              'grouping_module_sha256': hashlib.sha256(Path(point2mae_grouping.__file__).read_bytes()).hexdigest(),
              'model_root': str(MODEL_ROOT), 'data_root': str(DATA_ROOT),
              'exclusion_directories': [str(p) for p in EXCLUSION_DIRS],
              'model_source_sha256': hashlib.sha256((MODEL_ROOT / 'models/Point_M2AE.py').read_bytes()).hexdigest(),
              'split_manifest_sha256': {s: hashlib.sha256((RUN_DIR / f'{s}.csv').read_bytes()).hexdigest() for s in splits}}
(RUN_DIR / 'config.json').write_text(json.dumps(RUN_CONFIG, indent=2))
(RUN_DIR / 'exclusion_audit.json').write_text(json.dumps(audit, indent=2))
print(json.dumps(audit, indent=2))
print('Saved manifests and configuration:', RUN_DIR)

train       :  21452 units; districts: {'LAM_2': 5642, 'NAR': 5666, 'LAM_1': 5627, 'UTU': 4517}
validation  :   2523 units; districts: {'UTU': 551, 'NAR': 664, 'LAM_2': 670, 'LAM_1': 638}
test        :   1263 units; districts: {'LAM_2': 334, 'NAR': 354, 'LAM_1': 313, 'UTU': 262}
Too-small clouds: [('LAM_2/SU_40780', 2), ('LAM_2/SU_41714', 7), ('LAM_2/SU_46132', 1)]
{
  "all_clouds": 28464,
  "exclusion_directory_counts": {
    "/home/khan365/Home_code/Point-M2AE-main/Output/full_20261006_001322_611477/LS_head_scarp_Slope_units": 1251,
    "/home/khan365/Home_code/Point-M2AE-main/Output/full_20261006_001322_611477/LS_Slope_units_without_head_scarp": 1972
  },
  "excluded_clouds": 3223,
  "eligible_clouds": 25241,
  "excluded_ids_missing_from_all": [],
  "too_small_clouds": 3,
  "retained_full_clouds": 25238,
  "retained_original_points": 118601348
}
Saved manifests and configuration: /home/khan365/Home_code/Point-M2AE-main/training_runs/pointm2ae_xy_20261030T190853Z_d323ddc4


## 3. Environment check — install later

Stop here until we configure the training environment. The later cells need
Python/Jupyter, NumPy, PyTorch with a compatible CUDA toolkit/GPU, `laspy`
(with a LAZ backend such as `lazrs` for compressed files), `timm`, `matplotlib`,
`tqdm`, and the local model's import dependencies. Its custom extensions are
`pointnet2_ops`, `knn_cuda`, and the compiled `chamfer` module. Use the nested
repository's README and requirements as the starting point when we set up
compatible versions; this notebook does not choose or install them automatically.

The supplied model explicitly allocates CUDA tensors; CPU execution is not
supported by this workflow. Restart the kernel after installing native extensions.

In [5]:
import importlib.util
import sys

required = ['numpy', 'torch', 'laspy', 'timm', 'matplotlib', 'tqdm',
            'pointnet2_ops', 'knn_cuda', 'chamfer', 'termcolor', 'yaml', 'easydict']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError('Environment not ready. Install later, then restart the kernel. Missing: ' + ', '.join(missing))

import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import laspy
from tqdm.auto import tqdm

if not torch.cuda.is_available():
    raise RuntimeError('This local Point-M2AE implementation requires a CUDA GPU.')
if CFG['cuda_device'] >= torch.cuda.device_count():
    raise ValueError('Requested CUDA device is not available.')
torch.cuda.set_device(CFG['cuda_device'])
DEVICE = torch.device('cuda', CFG['cuda_device'])
sys.path.insert(0, str(MODEL_ROOT))
import models.Point_M2AE as local_model_module
if Path(local_model_module.__file__).resolve() != (MODEL_ROOT / 'models/Point_M2AE.py').resolve():
    raise RuntimeError('Another models package is cached; restart the kernel to import the requested local model.')
Point_M2AE = local_model_module.Point_M2AE

random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
torch.cuda.manual_seed_all(CFG['seed'])
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
print('Local model:', local_model_module.__file__)
print('Device:', torch.cuda.get_device_name(DEVICE))
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)
# Native FPS/KNN/Chamfer kernels may still prevent bitwise reproducibility.
(RUN_DIR / 'environment.json').write_text(json.dumps({
    'python': sys.version, 'torch': str(torch.__version__), 'cuda': torch.version.cuda,
    'numpy': np.__version__, 'laspy': laspy.__version__,
    'gpu': torch.cuda.get_device_name(DEVICE)}, indent=2))

Local model: /home/khan365/Home_code/Point-M2AE-main/Point-M2AE-main/models/Point_M2AE.py
Device: NVIDIA RTX 4000 Ada Generation
PyTorch: 2.5.1+cu124 CUDA: 12.4


224

## 4. Read complete point clouds and subtract the centroid

Every XYZ record is passed to the model, in file order. There is **no point-count
limit, random point selection, cropping, tiling, or duplicate padding**, in any
split. LAS scaled XYZ coordinates are read in float64. For each complete cloud,
compute its centroid `(mean(X), mean(Y), mean(Z))` and subtract it from every
point before float32 conversion:

`(X_relative, Y_relative, Z_relative) = (X - mean(X), Y - mean(Y), Z - mean(Z))`

There is **no radius division or other rescaling**. The centroid becomes the
origin, while distances and the original coordinate units are preserved, up to
floating-point precision. The same preprocessing applies to every split.

The dataset verifies that the loaded point count matches the audited header.
Nonfinite coordinates and invalid point counts raise a path-specific error
rather than silently deleting points. Zero horizontal extent does not prevent
centroid subtraction. Files below eight
points were already audited and excluded before splitting.

DataLoaders return tensors of shape `[1, N, 3]`, with **N allowed to change for
every cloud**. One full cloud must fit in GPU memory; an out-of-memory error does
not trigger automatic downsampling. Gradient accumulation reduces the number of
clouds resident on the GPU, but cannot reduce one cloud's memory requirement.
No full-dataset cache is allocated.

In [6]:
class SlopeUnitDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        try:
            cloud = laspy.read(row['path'])
            xyz = np.asarray(cloud.xyz, dtype=np.float64)
            if xyz.ndim != 2 or xyz.shape[1] != 3 or len(xyz) != row['point_count']:
                raise ValueError('Loaded XYZ shape/count differs from the audited cloud; rerun data preparation.')
            if len(xyz) < 8:
                raise ValueError('Cloud has fewer than 8 original points.')
            if not np.isfinite(xyz).all():
                raise ValueError('Cloud contains nonfinite coordinates; points will not be silently removed.')
            # Express every XYZ coordinate relative to the full cloud centroid.
            # Centre in float64 before conversion; retain the original coordinate units.
            xyz -= xyz.mean(axis=0, keepdims=True)
            points = np.ascontiguousarray(xyz, dtype=np.float32)
            if not np.isfinite(points).all():
                raise ValueError('Centroid-relative points are not finite in float32.')
            assert len(points) == row['point_count'], 'Point preservation check failed.'
            return torch.from_numpy(points)
        except Exception as exc:
            raise RuntimeError(f'Failed to read/prepare {row["path"]}: {exc}') from exc

train_dataset = SlopeUnitDataset(splits['train'])
validation_dataset = SlopeUnitDataset(splits['validation'])
test_dataset = SlopeUnitDataset(splits['test'])
loader_generator = torch.Generator().manual_seed(CFG['seed'])
loader_options = dict(batch_size=1, num_workers=0, pin_memory=True, drop_last=False)
train_loader = DataLoader(train_dataset, shuffle=True, generator=loader_generator, **loader_options)
validation_loader = DataLoader(validation_dataset, shuffle=False, **loader_options)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_options)
example = validation_dataset[0]
assert example.shape == (splits['validation'][0]['point_count'], 3)
assert torch.equal(example, validation_dataset[0]), 'Full-cloud loading changed.'
print('Full example shape:', tuple(example.shape), 'dtype:', example.dtype)

Full example shape: (8966, 3) dtype: torch.float32


## 5. Local Point-M2AE with XY-only positional embeddings

The subclass below calls the supplied model constructor and replaces **all three
encoder and both decoder positional MLPs**. Each replacement slices `[..., :2]`
and has `Linear(2, width)`. It does not merely zero the Z input. The upstream
XYZ reconstruction head is inherited. Before every upstream forward pass, `point2mae_grouping.py`
computes centre counts from the current cloud size using successive ceiling
division by `group_divisors`. Counts have minima `[8, 4, 3]` and cannot exceed
the available points/centres. Neighbour sizes shrink when necessary to fit
small clouds and retain masked reconstruction targets. With default settings,
16 points produce `[8, 4, 3]` centres and `[8, 4, 2]` neighbours; 61,907 points
produce `[7739, 3870, 968]` centres and `[8, 4, 4]` neighbours.

Later levels group centres from the previous level. FPS/KNN neighbourhoods
can overlap, so this is not disjoint spatial partitioning. All original points
are supplied as input, but not every point is guaranteed to enter a selected
neighbourhood. The nested model sources are unchanged; keep the grouping
module beside this notebook when reloading checkpoints. Training starts from scratch.

The checks verify that changing only Z cannot change positional embeddings,
that Z receives zero positional gradient, and that all five first linear layers
have two inputs. GPU smoke checks then verify finite reconstruction losses and gradients for
smallest → largest → smallest training clouds, without updating weights or BatchNorm
statistics. These checks and training require the configured CUDA environment.

In [7]:
from types import SimpleNamespace
from point2mae_grouping import hierarchy_for_cloud, configure_group_dividers

class XYPositionEmbedding(nn.Sequential):
    def __init__(self, width):
        super().__init__(nn.Linear(2, width), nn.GELU(), nn.Linear(width, width))

    def forward(self, centers):
        return super().forward(centers[..., :2])

class PointM2AEXY(Point_M2AE):
    def __init__(self, config):
        # Upstream constructor placeholders; counts are recalculated on every forward.
        initial_groups, _ = hierarchy_for_cloud(
            8, config.group_divisors, config.group_sizes, config.mask_ratio)
        upstream_config = SimpleNamespace(**{**vars(config), 'num_groups': initial_groups})
        super().__init__(upstream_config)
        self.h_encoder.encoder_pos_embeds = nn.ModuleList([
            XYPositionEmbedding(width) for width in config.encoder_dims])
        self.decoder_pos_embeds = nn.ModuleList([
            XYPositionEmbedding(width) for width in config.decoder_dims])
        self.h_encoder.encoder_pos_embeds.apply(self.h_encoder._init_weights)
        self.decoder_pos_embeds.apply(self.h_encoder._init_weights)

    def forward(self, pts, eval=False, **kwargs):
        if pts.ndim != 3 or pts.shape[0] != 1 or pts.shape[-1] != 3:
            raise ValueError('Full-cloud mode requires one cloud with shape [1, N, 3].')
        configure_group_dividers(
            self.group_dividers, pts.shape[1], self.config.group_divisors,
            self.config.group_sizes, self.config.mask_ratio)
        return super().forward(pts, eval=eval, **kwargs)

model = PointM2AEXY(SimpleNamespace(**MODEL_CONFIG)).to(DEVICE)
position_layers = list(model.h_encoder.encoder_pos_embeds) + list(model.decoder_pos_embeds)
assert len(position_layers) == 5
for embedding in position_layers:
    assert embedding[0].in_features == 2
    centers = torch.randn(2, 7, 3, device=DEVICE, requires_grad=True)
    changed = centers.detach().clone()
    changed[..., 2] += 1000
    torch.testing.assert_close(embedding(centers), embedding(changed), rtol=0, atol=0)
    gradient = torch.autograd.grad(embedding(centers).square().sum(), centers)[0]
    assert torch.count_nonzero(gradient[..., 2]).item() == 0
    assert torch.count_nonzero(gradient[..., :2]).item() > 0

# Exercise small -> large -> small transitions using training clouds only.
# eval() avoids updating BatchNorm statistics; backward checks gradients, not weights.
model.eval()
smallest = min(range(len(train_dataset)), key=lambda i: train_dataset.rows[i]['point_count'])
largest = max(range(len(train_dataset)), key=lambda i: train_dataset.rows[i]['point_count'])
for label, index in [('smallest training cloud', smallest), ('largest training cloud', largest),
                     ('smallest again', smallest)]:
    model.zero_grad(set_to_none=True)
    smoke_points = train_dataset[index].unsqueeze(0).to(DEVICE)
    smoke_loss = model(smoke_points, eval=False)
    assert smoke_loss.ndim == 0 and torch.isfinite(smoke_loss).item()
    smoke_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), float('inf'), error_if_nonfinite=True)
    expected_groups, expected_sizes = hierarchy_for_cloud(
        smoke_points.shape[1], CFG['group_divisors'], CFG['group_sizes'], CFG['mask_ratio'])
    assert [d.num_group for d in model.group_dividers] == expected_groups
    assert [d.group_size for d in model.group_dividers] == expected_sizes
    print(label, 'points:', smoke_points.shape[1], 'groups:', expected_groups,
          'neighbours:', expected_sizes, 'loss:', smoke_loss.item())
    model.zero_grad(set_to_none=True)
    del smoke_points, smoke_loss
print('XY-only positional and full-cloud forward/backward checks passed.')
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))


2026-10-30 15:09:25,395 - Point_M2AE - INFO - [Point_M2AE]


smallest training cloud points: 16 groups: [8, 4, 3] neighbours: [8, 4, 2] loss: 1.3411425352096558
largest training cloud points: 61907 groups: [7739, 3870, 968] neighbours: [8, 4, 4] loss: 0.8842653036117554
smallest again points: 16 groups: [8, 4, 3] neighbours: [8, 4, 2] loss: 1.2085669040679932
XY-only positional and full-cloud forward/backward checks passed.
Trainable parameters: 15249976


## 6. Train and select the best validation checkpoint

Execute the next two cells to train. AdamW and cosine learning-rate decay operate
in float32; the learning rate is logged before each epoch's scheduler step.
Validation uses reconstruction mode (`eval=False` argument) with `model.eval()`;
the upstream `eval=True` argument returns embeddings, not a reconstruction loss.
A fixed RNG context gives the same validation masks each epoch without disturbing
the training random stream. Each complete cloud contributes equally to the loss,
regardless of its point count. Gradient accumulation averages cloud losses over
`accumulation_steps`; a final shorter window uses its actual number of clouds.
Clipping and optimizer updates occur only at window boundaries. Losses below
are mean per-cloud squared Chamfer losses in the original coordinate units
squared (m² when XYZ is in metres); they are not elevation RMSE. These loss
values are not directly comparable with runs that divided by the XY radius.

`best.pt` is selected by validation loss only. `last.pt` stores the latest completed
epoch, optimizer, and scheduler state; `history.csv` stores losses and learning
rates. Early stopping is optional. If interrupted, the last completed checkpoint
remains. Re-executing the training cell continues the current in-memory model with
a fresh optimizer: for a fresh experiment, rerun from settings and recreate the
model. The held-out test set is evaluated separately after model selection.

In [8]:
from contextlib import contextmanager

@contextmanager
def fixed_evaluation_rng(seed):
    python_state, numpy_state = random.getstate(), np.random.get_state()
    try:
        with torch.random.fork_rng(devices=[DEVICE.index]):
            random.seed(seed)
            np.random.seed(seed)
            torch.random.default_generator.manual_seed(seed)
            with torch.cuda.device(DEVICE):
                torch.cuda.manual_seed(seed)
            yield
    finally:
        random.setstate(python_state)
        np.random.set_state(numpy_state)

def evaluate_reconstruction(model, loader, seed):
    previous_mode = model.training
    model.eval()
    total, samples = 0.0, 0
    try:
        with fixed_evaluation_rng(seed), torch.no_grad():
            for points in loader:
                points = points.to(DEVICE, non_blocking=True)
                loss = model(points, eval=False)
                if not torch.isfinite(loss).item():
                    raise FloatingPointError('Nonfinite evaluation loss.')
                total += loss.item() * len(points)
                samples += len(points)
    finally:
        model.train(previous_mode)
    if not samples:
        raise ValueError('Evaluation split is empty.')
    return total / samples

def save_checkpoint(path, state):
    temporary = path.with_suffix('.tmp')
    torch.save(state, temporary)
    temporary.replace(path)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['learning_rate'], weight_decay=CFG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs'])
history = []
best_validation = float('inf')
no_improvement = 0

for epoch in range(1, CFG['epochs'] + 1):
    model.train()
    train_sum, train_count = 0.0, 0
    learning_rate = optimizer.param_groups[0]['lr']
    progress = tqdm(train_loader, desc=f'Epoch {epoch}/{CFG["epochs"]}')
    optimizer.zero_grad(set_to_none=True)
    for cloud_step, points in enumerate(progress):
        points = points.to(DEVICE, non_blocking=True)
        loss = model(points, eval=False)
        if not torch.isfinite(loss).item():
            raise FloatingPointError(f'Nonfinite training loss in epoch {epoch}.')
        divisor = accumulation_window_size(cloud_step, len(train_loader), CFG['accumulation_steps'])
        (loss / divisor).backward()
        at_boundary = ((cloud_step + 1) % CFG['accumulation_steps'] == 0
                       or cloud_step + 1 == len(train_loader))
        if at_boundary:
            # Check gradient finiteness even when clipping is disabled.
            torch.nn.utils.clip_grad_norm_(model.parameters(),
                CFG['gradient_clip'] if CFG['gradient_clip'] > 0 else float('inf'), error_if_nonfinite=True)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
        train_sum += loss.item() * len(points)
        train_count += len(points)
        progress.set_postfix(chamfer=f'{train_sum / train_count:.6f}')
    validation_loss = evaluate_reconstruction(model, validation_loader, CFG['seed'])
    row = {'epoch': epoch, 'train_chamfer': train_sum / train_count,
           'validation_chamfer': validation_loss, 'learning_rate': learning_rate}
    history.append(row)
    scheduler.step()
    improved = validation_loss < best_validation
    if improved:
        best_validation, no_improvement = validation_loss, 0
    else:
        no_improvement += 1
    state = {'epoch': epoch, 'model_state_dict': model.state_dict(),
             'optimizer_state_dict': optimizer.state_dict(), 'scheduler_state_dict': scheduler.state_dict(),
             'best_validation_chamfer': best_validation, 'history': history, 'config': RUN_CONFIG}
    save_checkpoint(RUN_DIR / 'last.pt', state)
    if improved:
        save_checkpoint(RUN_DIR / 'best.pt', state)
    write_csv(RUN_DIR / 'history.csv', history,
              ['epoch', 'train_chamfer', 'validation_chamfer', 'learning_rate'])
    print(f'Epoch {epoch}: train={row["train_chamfer"]:.6f}, validation={validation_loss:.6f}, best={best_validation:.6f}')
    if CFG['patience'] and no_improvement >= CFG['patience']:
        print('Early stopping: validation loss did not improve.')
        break

print('Best checkpoint:', RUN_DIR / 'best.pt')

Epoch 1/100:   0%|          | 0/21452 [00:00<?, ?it/s]

Epoch 1: train=1.635338, validation=1.865408, best=1.865408


Epoch 2/100:   0%|          | 0/21452 [00:00<?, ?it/s]

Epoch 2: train=1.517107, validation=1.828166, best=1.828166


Epoch 3/100:   0%|          | 0/21452 [00:00<?, ?it/s]

Epoch 3: train=1.476370, validation=1.850478, best=1.828166


... (3 outputs left out)

Epoch 5: train=1.434025, validation=1.763594, best=1.763594


Epoch 6/100:   0%|          | 0/21452 [00:00<?, ?it/s]

Epoch 6: train=1.414202, validation=1.852622, best=1.763594


Epoch 7/100:   0%|          | 0/21452 [00:00<?, ?it/s]

## 7. Evaluate the held-out test split once

Load the validation-selected checkpoint and evaluate every point of each retained
test cloud as input, one cloud at a time, using fixed masks. Test results do not
select checkpoints or control early stopping. The score is an equal-cloud mean
of masked hierarchical reconstruction losses, not a dense full-cloud reconstruction
metric. The model's internal hierarchy still selects patch centres/neighbourhoods.

In [ ]:
checkpoint_path = RUN_DIR / 'best.pt'
if not checkpoint_path.is_file():
    raise FileNotFoundError('Run training before evaluating the best checkpoint.')
# Only load the checkpoint produced by this run. State contains tensors and basic Python values.
checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'], strict=True)
test_chamfer = evaluate_reconstruction(model, test_loader, CFG['seed'])
metrics = {'best_epoch': checkpoint['epoch'],
           'best_validation_chamfer': checkpoint['best_validation_chamfer'],
           'test_chamfer': test_chamfer, 'test_units': len(test_dataset),
           'evaluation_seed': CFG['seed'], 'batch_size': 1, 'input_mode': 'full_cloud',
           'test_original_points': sum(row['point_count'] for row in splits['test'])}
(RUN_DIR / 'test_metrics.json').write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))

In [ ]:
import matplotlib.pyplot as plt

with (RUN_DIR / 'history.csv').open(newline='') as handle:
    saved_history = list(csv.DictReader(handle))
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot([int(r['epoch']) for r in saved_history], [float(r['train_chamfer']) for r in saved_history], label='Train')
ax.plot([int(r['epoch']) for r in saved_history], [float(r['validation_chamfer']) for r in saved_history], label='Validation')
ax.set(xlabel='Epoch', ylabel='Masked XYZ Chamfer-L2', title='Point-M2AE — XY-only positional embeddings')
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(RUN_DIR / 'loss_curve.png', dpi=160)
plt.show()

## Artifacts and reuse

Each run in `training_runs/pointm2ae_xy_<timestamp>_<id>/` contains:

- `config.json`, `environment.json`, and `exclusion_audit.json` for provenance.
- `train.csv`, `validation.csv`, `test.csv`, and `excluded.csv` for auditing the data split.
- `too_small_clouds.csv` for the explicitly excluded clouds below eight points.
- `cloud_inventory.csv` for original point counts and adapted hierarchy settings.
- `best.pt`, `last.pt`, `history.csv`, `loss_curve.png`, and `test_metrics.json` after training/evaluation.

For inference in a later session, run the imports and XY model class definitions,
load `config.json`, instantiate `PointM2AEXY(SimpleNamespace(**saved_config['model']))`,
and load `model_state_dict` with `strict=True`. Use the same full-cloud loading, ratio hierarchy, and
centroid-only preprocessing. Use the preprocessing recorded with each checkpoint;
older radius-scaled checkpoints require their original preprocessing. Keep `point2mae_grouping.py` on the import path and restore the saved
`group_divisors` with the model configuration. Older fixed-count configurations
require an explicit choice of divisors for this adapter. Standard upstream XYZ positional checkpoints are not directly
compatible with these two-input positional layers.

This notebook was created without installing the environment or running GPU
training. Its JSON, Python cell syntax, and split/exclusion logic can be checked
independently. The forward smoke check, positional gradient checks, and full
training validation must be run once the CUDA environment is available.